# Hands-on Challenge: Build a Mini Transformer from Scratch

**Neural Networks — Postgraduate Course**  
**University of São Paulo · USP Robotics Center (CRoB)**  
**Instructor:** Dr. Ricardo Vilela de Godoy

---

## The task (15:50 → 17:30)

You are going to build a **character-level Transformer language model**, train it on Shakespeare, and make it generate text. Along the way, you will implement the three pieces that make a Transformer a Transformer:

1. **Self-attention** — the scaled dot-product
2. **Multi-head attention** — several attention heads in parallel
3. **The Transformer block** — attention + feed-forward + residuals + LayerNorm

Most of the code is done for you. Your job is to fill in the `# TODO` blocks. Each `TODO` is a tiny function, so the whole challenge is under ~80 lines of code you have to write.

## Rules of engagement

- **Work in pairs** — robotics is a team sport
- **Use Runtime → Change runtime type → T4 GPU** before running anything. The model trains in ~3 minutes on a T4.
- **Don't copy `nn.MultiheadAttention`** — the whole point is to understand what's inside.
- **Ask questions.** I will be walking around.

## Deliverables (by 17:30)

1. All `TODO` cells implemented and the shape tests passing.
2. A training run that brings validation loss below **~1.7**.
3. A sample of generated text (pasted in the final cell).
4. Your answers to the three reflection questions at the end.



---

## 0 · Setup

Run this cell first. It imports PyTorch, sets a seed, and picks the GPU if available.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

## 1 · Data: Tiny Shakespeare

We'll use the classic Tiny Shakespeare dataset (~1MB of the Bard's plays, concatenated). It is small enough to train on in minutes and large enough to produce surprisingly Shakespearean output.

In [ ]:
import urllib.request

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'input.txt')

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Dataset length: {len(text):,} characters')
print('--- first 250 characters ---')
print(text[:250])

### 1.1 · Character-level tokenizer

The simplest possible tokenizer: one token per character. The vocabulary is just the set of unique characters in the corpus.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
print(f'Chars: {"".join(chars)}')

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

print("\nencode('Hello'): ", encode('Hello'))
print("decode([20, 47, 1]): ", decode([20, 47, 1]))

In [ ]:
# Encode the entire dataset as a long tensor of token ids, split 90/10
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f'Train tokens: {len(train_data):,}  |  Val tokens: {len(val_data):,}')

### 1.2 · Hyperparameters

Small enough to train quickly, large enough to learn something. Feel free to tweak after your first working run.

In [ ]:
block_size = 128   # max context length (tokens the model sees at once)
batch_size = 64    # parallel sequences per step
n_embd     = 192   # embedding dimension  (d_model)
n_head     = 6     # number of attention heads
n_layer    = 4     # number of Transformer blocks stacked
dropout    = 0.1

assert n_embd % n_head == 0, 'n_embd must be divisible by n_head'
head_size = n_embd // n_head
print(f'Per-head dimension (d_k): {head_size}')

In [ ]:
def get_batch(split):
    """Sample a batch of (x, y) pairs. y is x shifted by one — next-char prediction."""
    src = train_data if split == 'train' else val_data
    ix = torch.randint(len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('x shape:', xb.shape, '(batch_size, block_size)')
print('y shape:', yb.shape, '(shifted by one)')

---

## 2 · Challenge 1 — Single-head self-attention  

Time to implement the core operation from the lecture:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}} + M\right) V$$

where $M$ is the causal mask (−∞ on future positions, 0 elsewhere).

### Your task

Fill in the `forward` method of `SingleHeadAttention` below. Follow the steps in the comments. The required output shape is `(B, T, head_size)`.

**Hints**
- `x.shape` gives you `(B, T, C)` where `B`=batch, `T`=tokens, `C`=`n_embd`.
- `self.key`, `self.query`, `self.value` are `nn.Linear(n_embd, head_size)` layers — just call them.
- For the matrix multiply, `@` or `torch.matmul` both work.
- Use `self.tril[:T, :T]` to get the causal mask for the current sequence length.
- `masked_fill(mask == 0, float('-inf'))` zeros out future positions before the softmax.

In [ ]:
class SingleHeadAttention(nn.Module):
    """One head of scaled dot-product self-attention with a causal mask."""

    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # The causal mask — a lower-triangular matrix of 1s
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        # ====================================================================
        # TODO 1 — SCALED DOT-PRODUCT ATTENTION
        # ====================================================================
        # Step 1: project x into keys, queries, values.
        #    k, q, v  →  each of shape (B, T, head_size)
        k = None  # TODO
        q = None  # TODO
        v = None  # TODO

        # Step 2: compute attention scores.
        #    scores = q @ k^T  →  shape (B, T, T)
        #    Remember to transpose only the last two dims of k.
        scores = None  # TODO

        # Step 3: scale by sqrt(d_k).
        scores = None  # TODO

        # Step 4: apply the causal mask.
        #    Use self.tril[:T, :T] and masked_fill with float('-inf').
        scores = None  # TODO

        # Step 5: softmax over the last dimension → attention weights.
        weights = None  # TODO
        weights = self.dropout(weights)

        # Step 6: weighted sum of values.  out shape: (B, T, head_size)
        out = None  # TODO

        return out
        # ====================================================================

# --- shape test ---
head = SingleHeadAttention(head_size).to(device)
dummy = torch.randn(2, block_size, n_embd, device=device)
try:
    out = head(dummy)
    assert out.shape == (2, block_size, head_size), f'Wrong shape: {out.shape}'
    print(f'✅  SingleHeadAttention output shape: {tuple(out.shape)}')
except Exception as e:
    print(f'❌  Something is wrong: {e}')

### 2.1 · Sanity check — visualize the attention pattern

Once your `SingleHeadAttention` works, the cell below will plot the attention weights for a single sequence. Because of the causal mask, the matrix must be **lower-triangular** — every position only attends to itself and earlier positions. If you see a full square, your mask is broken.

In [ ]:
import matplotlib.pyplot as plt

class _InspectHead(SingleHeadAttention):
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        return F.softmax(scores, dim=-1)

inspect = _InspectHead(head_size).to(device)
with torch.no_grad():
    W = inspect(torch.randn(1, 32, n_embd, device=device))[0].cpu()

plt.figure(figsize=(5, 5))
plt.imshow(W, cmap='viridis')
plt.title('Causal attention pattern (should be lower-triangular)')
plt.xlabel('key position'); plt.ylabel('query position')
plt.colorbar()
plt.show()

---

## 3 · Challenge 2 — Multi-head attention  

One head is nice. Several heads in parallel are nicer, as each can learn a different relationship.

### Your task

Implement `MultiHeadAttention` below. It should:

1. Run `n_head` independent `SingleHeadAttention` modules on the same input.
2. Concatenate their outputs along the last (feature) dimension.
3. Project back to `n_embd` with a linear layer.

**Hints**
- Use `nn.ModuleList` to hold the heads — a plain Python list will not register parameters.
- `torch.cat([...], dim=-1)` concatenates along the last dimension.
- After concat, the shape will be `(B, T, n_head * head_size)` = `(B, T, n_embd)`.

In [ ]:
class MultiHeadAttention(nn.Module):
    """n_head parallel self-attention heads, concatenated and projected."""

    def __init__(self, n_head, head_size):
        super().__init__()
        # ====================================================================
        # TODO 2 — MULTI-HEAD ATTENTION — __init__
        # ====================================================================
        # Create:
        #   self.heads : an nn.ModuleList of n_head SingleHeadAttention modules
        #   self.proj  : nn.Linear(n_head * head_size, n_embd) — the output projection W_O
        #   self.dropout : nn.Dropout(dropout)
        self.heads   = None  # TODO
        self.proj    = None  # TODO
        self.dropout = None  # TODO
        # ====================================================================

    def forward(self, x):
        # ====================================================================
        # TODO 2 — MULTI-HEAD ATTENTION — forward
        # ====================================================================
        # Step 1: run every head on x, concatenate results along the last dim.
        out = None  # TODO

        # Step 2: project back to n_embd and apply dropout.
        out = None  # TODO

        return out
        # ====================================================================

# --- shape test ---
mha = MultiHeadAttention(n_head, head_size).to(device)
dummy = torch.randn(2, block_size, n_embd, device=device)
try:
    out = mha(dummy)
    assert out.shape == (2, block_size, n_embd), f'Wrong shape: {out.shape}'
    print(f'✅  MultiHeadAttention output shape: {tuple(out.shape)}')
except Exception as e:
    print(f'❌  Something is wrong: {e}')

---

## 4 · Challenge 3 — The Transformer block  

A single Transformer block is:

```
x → LayerNorm → MultiHeadAttention → (+ residual)
  → LayerNorm → FeedForward        → (+ residual)
```

We use the **pre-norm** variant (LayerNorm *before* the sublayer) — it is much easier to train than the original post-norm design.

The feed-forward network is a tiny MLP: `Linear(d → 4d) → GELU → Linear(4d → d) → Dropout`.

### Your task

Fill in the `forward` of `TransformerBlock`. **Don't forget the residual connections** — `x = x + sublayer(norm(x))`.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise MLP: d_model → 4·d_model → d_model."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm Transformer block: attention + FFN with residuals."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.attn = MultiHeadAttention(n_head, head_size)
        self.ffn  = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        # ====================================================================
        # TODO 3 — PRE-NORM TRANSFORMER BLOCK
        # ====================================================================
        # Apply LayerNorm, then attention, then add the residual.
        # Then LayerNorm, then feed-forward, then add the residual.
        x = None  # TODO: x + attn(ln1(x))
        x = None  # TODO: x + ffn(ln2(x))
        return x
        # ====================================================================

# --- shape test ---
block = TransformerBlock(n_embd, n_head).to(device)
dummy = torch.randn(2, block_size, n_embd, device=device)
try:
    out = block(dummy)
    assert out.shape == (2, block_size, n_embd), f'Wrong shape: {out.shape}'
    print(f'✅  TransformerBlock output shape: {tuple(out.shape)}')
except Exception as e:
    print(f'❌  Something is wrong: {e}')

---

## 5 · The full model (given)

You already wrote the hard parts. The full model just wires together:

- Token embedding table  `(vocab_size → n_embd)`
- **Learned** positional embedding table  `(block_size → n_embd)` — simpler than sinusoidal
- `n_layer` stacked `TransformerBlock`s
- A final LayerNorm
- A linear head projecting to `vocab_size` logits

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[TransformerBlock(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.head    = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)                                      # (B, T, C)
        pos = self.pos_emb(torch.arange(T, device=idx.device))       # (T, C)
        x = tok + pos                                                # broadcast
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)                                        # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]                          # crop to context
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature                  # last step only
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx

model = MiniGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params/1e6:.2f} M')

## 6 · Training loop

AdamW, ~3000 steps, evaluating every 300 steps. Should take roughly 2–4 minutes on a T4.

**Target:** validation loss below **~1.7**. Random guessing on 65 characters would give ≈ 4.17.

In [ ]:
max_iters = 3000
eval_interval = 300
eval_iters = 50
learning_rate = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

history = []
for step in range(max_iters + 1):
    if step % eval_interval == 0 or step == max_iters:
        losses = estimate_loss()
        history.append((step, losses['train'], losses['val']))
        print(f'step {step:5d}  |  train loss {losses["train"]:.4f}  |  val loss {losses["val"]:.4f}')

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('\n✅  Training done.')

In [ ]:
# Loss curve
steps, tr, va = zip(*history)
plt.figure(figsize=(7, 4))
plt.plot(steps, tr, label='train', marker='o')
plt.plot(steps, va, label='val',   marker='s')
plt.xlabel('step'); plt.ylabel('cross-entropy loss')
plt.title('MiniGPT on Tiny Shakespeare')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 7 · Generate Shakespeare

Time to see what the model has learned. Feed it a seed and let it generate 500 new characters.

In [ ]:
seed = 'ROMEO:'
ctx = torch.tensor([encode(seed)], dtype=torch.long, device=device)
out = model.generate(ctx, max_new_tokens=500, temperature=0.9, top_k=40)[0].tolist()
print(decode(out))

---

## 8 · Reflection questions  

Answer briefly (2–4 sentences each) in the markdown cell below. Submit with the rest of your notebook.

1. **Why does scaled dot-product attention divide by $\sqrt{d_k}$?** What would go wrong if we did not?
2. **What exactly does the causal mask prevent?** If you removed it and trained on the same data, what would break at generation time and why?
3. **Identify two concrete limits of this tiny model.** What architectural or data change would you try first to make it better, and what do you expect to improve?

### Your answers

**1.** _Your answer here._

**2.** _Your answer here._

**3.** _Your answer here._

---

## 9 · Bonus challenges  ⭐ (if you finish early)

Pick any of these. None are required for full credit, but each one will teach you something real.

- **⭐ Sinusoidal positional encoding.** Replace `self.pos_emb` (a learned table) with the original Vaswani sinusoidal formula. Does it train as well? Does it let you generate beyond `block_size`?
- **⭐⭐ Batched multi-head attention.** The current `MultiHeadAttention` loops over heads in Python. Rewrite it with a single `(B, n_head, T, head_size)` tensor — this is how real implementations do it. Measure the speedup.
- **⭐⭐ Temperature & top-k sweep.** Generate 200 chars for `temperature ∈ {0.5, 0.8, 1.0, 1.3}` and `top_k ∈ {None, 10, 40}`. Which combo gives the most Shakespearean output? Which gives the most chaos?
- **⭐⭐⭐ Your own dataset.** Swap Tiny Shakespeare for something else — code, Portuguese text, a Petrobras inspection manual, song lyrics. Does the model pick up domain-specific patterns?
- **⭐⭐⭐ Attention heatmaps on a real sentence.** Encode a sentence, run it through the trained model, and visualize the attention weights of one head in the last layer. Do any heads look interpretable?

---

## Good luck
